In [ ]:
import json
import numpy as np
import pandas as pd
from config import CSV_PATH, OUT

raw = pd.read_csv('amazon_delivery.csv')
df = raw.copy()
print("Строк:", len(df), " Столбцов:", df.shape[1])

bad_coords = (df["Store_Latitude"] <= 1) | (df["Store_Longitude"] <= 1)
quality = {
    "заказов": len(df),
    "дубликаты Order_ID": int(df["Order_ID"].duplicated().sum()),
    "пропусков Weather": int(df["Weather"].isna().sum()),
    "пропусков Traffic": int(df["Traffic"].astype(str).str.strip().isin(["NaN", "nan"]).sum()),
    "пропусков Agent_Rating": int(df["Agent_Rating"].isna().sum()),
    "рейтинг больше 5": int((df["Agent_Rating"] > 5).sum()),
    "неверные координаты магазина": int(bad_coords.sum()),
}

text_cols = ["Weather", "Traffic", "Vehicle", "Area", "Category"]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace("NaN", np.nan)
df["Area"] = df["Area"].replace("Metropolitian", "Metropolitan")

df["Order_Date"] = pd.to_datetime(df["Order_Date"])
order_time = pd.to_datetime(df["Order_Time"].astype(str).str.strip(), format="%H:%M:%S", errors="coerce")
pickup_time = pd.to_datetime(df["Pickup_Time"].astype(str).str.strip(), format="%H:%M:%S", errors="coerce")
quality["строк без времени заказа"] = int(order_time.isna().sum())
quality["строк без времени забора"] = int(pickup_time.isna().sum())

has_time = order_time.notna()
df, order_time, pickup_time = df[has_time].copy(), order_time[has_time], pickup_time[has_time]
bad_coords = bad_coords[has_time]

df["order_hour"] = order_time.dt.hour
df["weekday"] = df["Order_Date"].dt.dayofweek
df["is_weekend"] = (df["weekday"] >= 5).astype(int)
df["pickup_delay_min"] = ((pickup_time - order_time).dt.total_seconds() / 60) % 1440

lat1 = np.radians(df["Store_Latitude"])
lon1 = np.radians(df["Store_Longitude"])
lat2 = np.radians(df["Drop_Latitude"])
lon2 = np.radians(df["Drop_Longitude"])
a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
df["distance_km"] = 2 * 6371 * np.arcsin(np.sqrt(a))

df["coord_invalid"] = bad_coords.astype(int)
df.loc[bad_coords, "distance_km"] = np.nan

df["store_id"] = df["Store_Latitude"].round(3).astype(str) + "_" + df["Store_Longitude"].round(3).astype(str)
df.loc[bad_coords, "store_id"] = "unknown"

for col in ["Agent_Rating", "distance_km", "pickup_delay_min"]:
    df[col] = df[col].fillna(df[col].median())
for col in text_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

df.to_csv(f"{OUT}/clean_data.csv", index=False)
quality["заказов после очистки"] = len(df)
quality["дней в данных"] = int(df["Order_Date"].nunique())
quality["магазинов с верными координатами"] = int(df.loc[~bad_coords, "store_id"].nunique())
quality["период"] = f"{df['Order_Date'].min().date()} — {df['Order_Date'].max().date()}"
with open(f"{OUT}/quality.json", "w", encoding="utf-8") as f:
    json.dump(quality, f, ensure_ascii=False, indent=2)

print("\nОтчёт о качестве данных:")
for k, v in quality.items():
    print(f"  {k}: {v}")
print("\nГотово: outputs/clean_data.csv")


Строк: 43739  Столбцов: 16

Отчёт о качестве данных:
  заказов: 43739
  дубликаты Order_ID: 0
  пропусков Weather: 91
  пропусков Traffic: 91
  пропусков Agent_Rating: 54
  рейтинг больше 5: 53
  неверные координаты магазина: 3693
  строк без времени заказа: 91
  строк без времени забора: 0
  заказов после очистки: 43648
  дней в данных: 44
  магазинов с верными координатами: 364
  период: 2022-02-11 — 2022-04-06

Готово: outputs/clean_data.csv


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
from config import OUT, LATE_MIN, TARGET, load_clean, make_X, split_by_date

df = load_clean()
time = df[TARGET]

print("Время доставки, минуты:")
print("  среднее:", round(time.mean(), 1), " медиана:", time.median(),
      " мин:", time.min(), " макс:", time.max())
late_share = (time > LATE_MIN).mean() * 100
peak_share = df["order_hour"].between(17, 23).mean() * 100
print(f"  дольше {LATE_MIN} мин: {late_share:.1f}% заказов")
print(f"  заказов с 17:00 до 23:59: {peak_share:.1f}%")

print("\nСреднее время по трафику:\n", df.groupby("Traffic")[TARGET].mean().round(1))
print("\nСреднее время по погоде:\n", df.groupby("Weather")[TARGET].mean().round(1))

orders_per_hour = df.groupby("order_hour")["Order_ID"].count()
plt.figure(figsize=(8, 4))
orders_per_hour.plot(kind="bar", color="steelblue")
plt.title("Количество заказов по часам")
plt.xlabel("Час заказа"); plt.ylabel("Заказов")
plt.tight_layout(); plt.savefig(f"{OUT}/chart1_orders_by_hour.png", dpi=120); plt.close()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df.groupby("Traffic")[TARGET].mean().sort_values().plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Среднее время доставки по трафику (мин)")
df.groupby("Weather")[TARGET].mean().sort_values().plot(kind="barh", ax=axes[1], color="steelblue")
axes[1].set_title("Среднее время доставки по погоде (мин)")
plt.tight_layout(); plt.savefig(f"{OUT}/chart2_traffic_weather.png", dpi=120); plt.close()

train, test = split_by_date(df)
X_train, y_train = make_X(train), train[TARGET]
X_test, y_test = make_X(test), test[TARGET]
print(f"\nОбучаем на {len(train)} заказах, проверяем на {len(test)} новых")

models = {
    "Линейная регрессия": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, min_samples_leaf=3, n_jobs=-1, random_state=42),
    "Gradient Boosting": HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=42),
}

table = []
baseline = np.full(len(y_test), y_train.median())
table.append({"модель": "Базовая (медиана)",
              "MAE": mean_absolute_error(y_test, baseline),
              "RMSE": np.sqrt(mean_squared_error(y_test, baseline)),
              "точно в пределах 20 мин, %": (abs(y_test - baseline) <= 20).mean() * 100})

fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fitted[name] = model
    table.append({"модель": name,
                  "MAE": mean_absolute_error(y_test, pred),
                  "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
                  "точно в пределах 20 мин, %": (abs(y_test - pred) <= 20).mean() * 100})

result = pd.DataFrame(table).round(2)
print("\nСравнение моделей (чем меньше MAE, тем лучше):")
print(result.to_string(index=False))
result.to_csv(f"{OUT}/model_comparison.csv", index=False)

best_name = result[result["модель"] != "Базовая (медиана)"].sort_values("MAE").iloc[0]["модель"]
base_mae = result.iloc[0]["MAE"]
best_mae = result[result["модель"] == best_name]["MAE"].iloc[0]
improve = (1 - best_mae / base_mae) * 100
print(f"\nЛучшая модель: {best_name}. Ошибка меньше базовой на {improve:.1f}%")

imp = permutation_importance(fitted[best_name], X_test, y_test, n_repeats=3,
                             scoring="neg_mean_absolute_error", random_state=42)
importance = pd.Series(imp.importances_mean, index=X_test.columns).sort_values(ascending=False)
print("\nТоп-8 факторов (на сколько минут растёт ошибка, если фактор испортить):")
print(importance.head(8).round(2))

plt.figure(figsize=(8, 4))
importance.head(8).sort_values().plot(kind="barh", color="steelblue")
plt.title("Что сильнее всего влияет на время доставки")
plt.tight_layout(); plt.savefig(f"{OUT}/chart3_importance.png", dpi=120); plt.close()

with open(f"{OUT}/ds_results.json", "w", encoding="utf-8") as f:
    json.dump({"среднее время, мин": round(time.mean(), 1),
               "мин": float(time.min()), "макс": float(time.max()),
               f"доля заказов дольше {LATE_MIN} мин, %": round(late_share, 1),
               "доля заказов 17-23 часа, %": round(peak_share, 1),
               "лучшая модель": best_name,
               "MAE базовой": base_mae, "MAE лучшей": best_mae,
               "улучшение, %": round(improve, 1),
               "в пределах 20 мин, %": float(result[result["модель"] == best_name]["точно в пределах 20 мин, %"].iloc[0])},
              f, ensure_ascii=False, indent=2)
print("\nГотово: графики и outputs/ds_results.json")


Время доставки, минуты:
  среднее: 124.9  медиана: 125.0  мин: 10  макс: 270
  дольше 160 мин: 23.1% заказов
  заказов с 17:00 до 23:59: 72.2%

Среднее время по трафику:
 Traffic
High      129.4
Jam       147.8
Low       101.4
Medium    126.8
Name: Delivery_Time, dtype: float64

Среднее время по погоде:
 Weather
Cloudy        138.3
Fog           136.6
Sandstorms    123.2
Stormy        123.2
Sunny         103.7
Windy         123.7
Name: Delivery_Time, dtype: float64

Обучаем на 34605 заказах, проверяем на 9043 новых

Сравнение моделей (чем меньше MAE, тем лучше):
            модель   MAE  RMSE  точно в пределах 20 мин, %
 Базовая (медиана) 41.78 52.49                       34.46
Линейная регрессия 25.40 32.25                       47.83
     Random Forest 17.99 23.26                       63.46
 Gradient Boosting 17.70 22.72                       63.45

Лучшая модель: Gradient Boosting. Ошибка меньше базовой на 57.6%

Топ-8 факторов (на сколько минут растёт ошибка, если фактор испортить

In [ ]:
import json
import joblib
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from config import OUT, LATE_MIN, TARGET, load_clean, make_X, split_by_date

df = load_clean()
train, test = split_by_date(df)
X_train, y_train = make_X(train), train[TARGET]
X_test, y_test = make_X(test), test[TARGET]

model = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
mae = mean_absolute_error(y_test, pred)
print(f"Средняя ошибка (MAE): {mae:.1f} мин")

really_late = y_test > LATE_MIN
said_late = pd.Series(pred, index=y_test.index) > LATE_MIN
precision = (really_late & said_late).sum() / said_late.sum() * 100   # из тех, что назвали долгими — сколько правда долгие
recall = (really_late & said_late).sum() / really_late.sum() * 100    # из всех долгих — сколько нашли
print(f"Долгие заказы: точность {precision:.0f}%, полнота {recall:.0f}%")

joblib.dump({"model": model, "columns": list(X_train.columns)}, f"{OUT}/model.joblib")


def predict_delivery_time(order):
    """order — словарь с данными заказа. Возвращает минуты."""
    saved = joblib.load(f"{OUT}/model.joblib")
    row = pd.DataFrame([order])
    X = make_X(row).reindex(columns=saved["columns"], fill_value=0)
    return float(saved["model"].predict(X)[0])


example = test.iloc[0].to_dict()
print("\nПример заказа:", {k: example[k] for k in ["Traffic", "Weather", "Vehicle", "distance_km"]})
print(f"Модель предсказывает: {predict_delivery_time(example):.0f} мин, "
      f"на самом деле: {example[TARGET]} мин")

with open(f"{OUT}/ml_results.json", "w", encoding="utf-8") as f:
    json.dump({"MAE, мин": round(mae, 2),
               "точность поиска долгих заказов, %": round(precision, 1),
               "полнота поиска долгих заказов, %": round(recall, 1)},
              f, ensure_ascii=False, indent=2)
print("\nГотово: outputs/model.joblib")


Средняя ошибка (MAE): 17.7 мин
Долгие заказы: точность 77%, полнота 86%

Пример заказа: {'Traffic': 'Medium', 'Weather': 'Sunny', 'Vehicle': 'motorcycle', 'distance_km': 7.790401360105752}
Модель предсказывает: 113 мин, на самом деле: 105 мин

Готово: outputs/model.joblib


In [ ]:
import json
import joblib
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from config import OUT, LATE_MIN, TARGET, load_clean, make_X, split_by_date

df = load_clean()

known = df[df["store_id"] != "unknown"]
hubs = known.groupby("store_id").agg(
    заказов=("Order_ID", "count"),
    среднее_время=(TARGET, "mean"),
)
hubs["долгих_заказов_%"] = known.groupby("store_id")[TARGET].apply(lambda s: (s > LATE_MIN).mean() * 100)
hubs = hubs.sort_values("заказов", ascending=False).round(1)
hubs.to_csv(f"{OUT}/hubs_load.csv")
print("Топ-5 самых загруженных хабов:")
print(hubs.head(5))

table = df.pivot_table(index="weekday", columns="order_hour", values="Order_ID", aggfunc="count")
plt.figure(figsize=(10, 3.5))
plt.imshow(table.values, aspect="auto", cmap="Blues")
plt.xticks(range(len(table.columns)), table.columns)
plt.yticks(range(7), ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"])
plt.colorbar(label="Заказов")
plt.title("Нагрузка: день недели × час")
plt.xlabel("Час заказа")
plt.tight_layout(); plt.savefig(f"{OUT}/chart4_heatmap.png", dpi=120); plt.close()

saved = joblib.load(f"{OUT}/model.joblib")
_, test = split_by_date(df)


def avg_prediction(orders):
    X = make_X(orders).reindex(columns=saved["columns"], fill_value=0)
    return saved["model"].predict(X).mean()


scenarios = {}
for traffic in ["Low", "Medium", "High", "Jam"]:
    changed = test.copy()
    changed["Traffic"] = traffic
    scenarios[f"Трафик = {traffic}"] = avg_prediction(changed)
for weather in ["Sunny", "Cloudy", "Stormy"]:
    changed = test.copy()
    changed["Weather"] = weather
    scenarios[f"Погода = {weather}"] = avg_prediction(changed)

print("\nСценарии «что если» (среднее предсказанное время доставки, мин):")
for name, minutes in scenarios.items():
    print(f"  {name}: {minutes:.0f}")

peak = df[df["order_hour"].between(17, 23)]
print(f"\nВывод для диспетчера: в 17-23 часа приходит {len(peak) / len(df) * 100:.0f}% заказов — "
      "к этому времени нужно выпускать больше курьеров.")

with open(f"{OUT}/twin_results.json", "w", encoding="utf-8") as f:
    json.dump({k: round(float(v), 1) for k, v in scenarios.items()}, f, ensure_ascii=False, indent=2)
print("\nГотово: outputs/hubs_load.csv, chart4_heatmap.png, twin_results.json")


Топ-5 самых загруженных хабов:
               заказов  среднее_время  долгих_заказов_%
store_id                                               
23.359_85.325      461          125.5              23.0
26.903_75.793      338          128.5              25.1
26.914_75.753      319          125.7              25.7
12.979_77.641      319          124.1              21.3
22.308_73.168      318          134.7              26.7

Сценарии «что если» (среднее предсказанное время доставки, мин):
  Трафик = Low: 108
  Трафик = Medium: 129
  Трафик = High: 136
  Трафик = Jam: 142
  Погода = Sunny: 109
  Погода = Cloudy: 141
  Погода = Stormy: 125

Вывод для диспетчера: в 17-23 часа приходит 72% заказов — к этому времени нужно выпускать больше курьеров.

Готово: outputs/hubs_load.csv, chart4_heatmap.png, twin_results.json


In [ ]:
import json
from config import OUT, LATE_MIN


def read(name):
    with open(f"{OUT}/{name}", encoding="utf-8") as f:
        return json.load(f)


quality = read("quality.json")     # от Нұрасыл
ds = read("ds_results.json")       # от Алдияр
ml = read("ml_results.json")       # от Асыл
twin = read("twin_results.json")   # от Шынғысхан

lines = [
    "ПАСПОРТ ПРОЕКТА — OpenEgiz Logistics",
    "=" * 50,
    "Объект: last-mile хаб (склад/магазин -> курьер -> клиент)",
    "Пользователь: менеджер хаба / диспетчер",
    f"Данные: {quality['заказов']} заказов, период {quality['период']}",
    "",
    "МЕРА ПРОБЛЕМЫ:",
    f"  - Среднее время доставки {ds['среднее время, мин']} мин (от {ds['мин']:.0f} до {ds['макс']:.0f})",
    f"  - Заказов дольше {LATE_MIN} мин: {ds[f'доля заказов дольше {LATE_MIN} мин, %']}%",
    f"  - Заказов в 17-23 часа: {ds['доля заказов 17-23 часа, %']}%",
    "",
    "ЗАДАЧА Data Science: прогноз времени доставки (регрессия)",
    f"  - Лучшая модель: {ds['лучшая модель']}",
    f"  - Ошибка: {ds['MAE базовой']} мин (базовая) -> {ds['MAE лучшей']} мин (модель), лучше на {ds['улучшение, %']}%",
    f"  - В пределах ±20 мин: {ds['в пределах 20 мин, %']}% заказов",
    f"  - Поиск долгих заказов: точность {ml['точность поиска долгих заказов, %']}%, полнота {ml['полнота поиска долгих заказов, %']}%",
    "",
    "СЦЕНАРИИ «ЧТО ЕСЛИ» (Digital Twin):",
]
for name, minutes in twin.items():
    lines.append(f"  - {name}: {minutes:.0f} мин")
lines += [
    "",
    "РИСКИ ДАННЫХ:",
    f"  - Неверные координаты магазина: {quality['неверные координаты магазина']} строк",
    f"  - Пропуски: погода {quality['пропусков Weather']}, трафик {quality['пропусков Traffic']}, рейтинг {quality['пропусков Agent_Rating']}",
    "  - Данные 2022 года и из Индии — для Казахстана нужны свои данные",
]

text = "\n".join(lines)
print(text)
with open(f"{OUT}/project_passport.txt", "w", encoding="utf-8") as f:
    f.write(text)


ПАСПОРТ ПРОЕКТА — OpenEgiz Logistics
Объект: last-mile хаб (склад/магазин -> курьер -> клиент)
Пользователь: менеджер хаба / диспетчер
Данные: 43739 заказов, период 2022-02-11 — 2022-04-06

МЕРА ПРОБЛЕМЫ:
  - Среднее время доставки 124.9 мин (от 10 до 270)
  - Заказов дольше 160 мин: 23.1%
  - Заказов в 17-23 часа: 72.2%

ЗАДАЧА Data Science: прогноз времени доставки (регрессия)
  - Лучшая модель: Gradient Boosting
  - Ошибка: 41.78 мин (базовая) -> 17.7 мин (модель), лучше на 57.6%
  - В пределах ±20 мин: 63.45% заказов
  - Поиск долгих заказов: точность 77.0%, полнота 86.3%

СЦЕНАРИИ «ЧТО ЕСЛИ» (Digital Twin):
  - Трафик = Low: 108 мин
  - Трафик = Medium: 129 мин
  - Трафик = High: 136 мин
  - Трафик = Jam: 142 мин
  - Погода = Sunny: 109 мин
  - Погода = Cloudy: 140 мин
  - Погода = Stormy: 124 мин

РИСКИ ДАННЫХ:
  - Неверные координаты магазина: 3693 строк
  - Пропуски: погода 91, трафик 91, рейтинг 54
  - Данные 2022 года и из Индии — для Казахстана нужны свои данные
